# Lab 4


In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("lab_4.ipynb")

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

<img src="cs88-logo.png" alt="CS88 Logo" style="width: 15%; float: right; padding: 1%; margin-right: 2%;"/>

## Lab 4: SQL


<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## SQL Basics

<hr style="border: 1px solid #fdb515;" />

First, let's set up SQLite and create a connection to an in-memory database:

In [ ]:
import sqlite3

# Create an in-memory SQLite database
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Load the data from data.sql
with open('data.sql', 'r') as f:
    sql_script = f.read()
    
# Execute the SQL script to create tables
cursor.executescript(sql_script)
conn.commit()

# Helper function to execute and display query results
def run_query(query):
    """Execute a SQL query and display the results."""
    # Check if this is a DDL statement (CREATE, DROP, ALTER, etc.)
    query_upper = query.strip().upper()
    is_ddl = query_upper.startswith(('CREATE', 'DROP', 'ALTER', 'INSERT', 'UPDATE', 'DELETE'))
    
    cursor.execute(query)
    
    # Commit DDL statements
    if is_ddl:
        conn.commit()
        # DDL statements don't return rows, so return empty list silently
        return []
    
    # For SELECT queries, fetch and display results
    results = cursor.fetchall()
    if results:
        # Get column names
        column_names = [description[0] for description in cursor.description]
        # Print header
        print('|'.join(column_names))
        print('-' * (sum(len(str(col)) for col in column_names) + len(column_names) - 1))
        # Print rows
        for row in results:
            print('|'.join(str(val) if val is not None else '' for val in row))
    else:
        print("(no results)")
    return results

print("Database loaded successfully!")

<hr style="border: 1px solid #fdb515;" />

### Creating Tables

You can create SQL tables either from scratch or from existing tables.

The following statement creates a table by specifying column names and values
without referencing another table. Each `SELECT` clause specifies the values
for one row, and `UNION` is used to join rows together. The `AS` clauses give a
name to each column; it need not be repeated in subsequent rows after the
first.

```sql
CREATE TABLE [table_name] AS
  SELECT [val1] AS [column1], [val2] AS [column2], ... UNION
  SELECT [val3]             , [val4]             , ... UNION
  SELECT [val5]             , [val6]             , ...;
```

Let's say we want to make the following table called `big_game` which records
the scores for the Big Game each year. This table has three columns:
`berkeley`, `stanford`, and `year`.

We could do so with the following `CREATE TABLE` statement:

In [ ]:

cursor.execute("""
    CREATE TABLE big_game AS
      SELECT 30 AS berkeley, 7 AS stanford, 2002 AS year UNION
      SELECT 28,             16,            2003         UNION
      SELECT 17,             38,            2014;
""")
conn.commit()


<hr style="border: 1px solid #fdb515;" />

### Selecting From Tables
More commonly, we will create new tables by selecting specific columns that we
want from existing tables by using a `SELECT` statement as follows:

```sql
SELECT [columns] FROM [tables] WHERE [condition] ORDER BY [columns] LIMIT [limit];
```

Let's break down this statement:

* `SELECT [columns]` tells SQL that we want to include the given columns in our
  output table; `[columns]` is a comma-separated list of column names, and `*`
  can be used to select all columns
* `FROM [table]` tells SQL that the columns we want to select are from the
  given table
* `WHERE [condition]` filters the output table by only including rows whose
  values satisfy the given `[condition]`, a boolean expression
* `ORDER BY [columns]` orders the rows in the output table by the given
  comma-separated list of columns
* `LIMIT [limit]` limits the number of rows in the output table by the integer
  `[limit]`

> *Note:* We capitalize SQL keywords purely because of style convention. It
> makes queries much easier to read, though they will still work if you don't
> capitalize keywords.

Here are some examples:

Select all of Berkeley's scores from the `big_game` table, but only include
scores from years past 2002:

In [ ]:
run_query("SELECT berkeley FROM big_game WHERE year > 2002")

Select the scores for both schools in years that Berkeley won:

In [ ]:
run_query("SELECT berkeley, stanford FROM big_game WHERE berkeley > stanford")

Select the years that Stanford scored more than 15 points:

In [ ]:
run_query("SELECT year FROM big_game WHERE stanford > 15")

<hr style="border: 1px solid #fdb515;" />

### SQL operators

Expressions in the `SELECT`, `WHERE`, and `ORDER BY` clauses can contain
one or more of the following operators:

* comparison operators: `=`, `>`, `<`, `<=`, `>=`, `<>` or `!=` ("not equal")
* boolean operators: `AND`, `OR`
* arithmetic operators: `+`, `-`, `*`, `/`
* concatenation operator: `||`

Here are some examples:

Output the ratio of Berkeley's score to Stanford's score each year:

In [ ]:
run_query("SELECT berkeley * 1.0 / stanford FROM big_game")

Output the sum of scores in years where both teams scored over 10 points:

In [ ]:
run_query("SELECT berkeley + stanford FROM big_game WHERE berkeley > 10 AND stanford > 10")

Output a table with a single column and single row containing the value "hello
world":

In [ ]:
run_query('SELECT "hello" || " " || "world"')

<hr style="border: 1px solid #fdb515;" />

## Joins

We can use *joins* to include rows from another table that satisfy the `WHERE`
predicate. Joins can either be on different tables, or the same table if we
include an alias. Here we are referencing the football table twice, once as
the alias `a` and once as the alias `b`.

```sql
SELECT a.Berkeley - b.Berkeley, a.Stanford - b.Stanford, a.Year, b.Year
FROM Football as a, Football as b WHERE a.Year > b.Year;

-11|22|2014|2003
-13|21|2014|2002
-2|9|2003|2002
```

What is this query asking for? We're creating a new table where each row is the difference
in scores from 2 years. For example, in the first row, we are comparing
the big game scores in 2014 and 2003. In 2014, Berkeley scored 17 and in 2003, Berkeley scored 28.
17 - 28 = -11. Similarly, in 2014, Stanford scored 38 and in 2003 they scored 16. 38 - 16 = 22.

You may notice that it does not seem like we actually performed any operations
to do the join. However, the join is implicit in the fact that we listed more
than one table after the FROM. In this example, we joined the table `Football`
with itself and gave each instance of the table an alias, `a` and `b` so that
we could distinctly refer to each table's attributes and perform selections and
comparisons on them, such as `a.Year > b.Year`.

One way to think of a join is that it produces a cross-product between the two
tables by matching each row from the first table with every other row in the
second table, which creates a new, larger joined table.

Here's an illustration of what happened in the joining process during the
above query.

![joins](joins.png)

From here, the select statement examines the joined table and selects the values
it desires: `a.Berkeley - b.Berkeley` and `a.Stanford - b.Stanford` but only
from the rows `WHERE a.Year > b.Year`. This prevents duplicate results from
appearing in our output!


### Getting to Know Students in Data C88C

In a past semester of Data C88C at UC Berkeley, we asked students to take a survey. In this lab, we will interact with the results of the survey by using SQL queries to see if we can find interesting trends in the data. Data scientists often use languages like SQL to perform exploratory data analysis.

First, take a look at `data.sql` and examine the tables defined in it. Note their structures. There are two tables you will be working with:


#### 1. `students`

The main results of the survey. Each column represents a different question from the survey, except for the first column, which is the time when the response was submitted. This timestamp is a unique identifier for each row in the table.

| Column Name | Question |
|------------|----------|
| `time` | The unique timestamp that identifies the submission |
| `number` | What’s your favorite number between 1 and 100? |
| `color` | What is your favorite color? |
| `seven` | Choose the number 7 below.<br>Options:<br>- 7<br>- You are not the boss of me!<br>- I do what I want.<br>- I'm a rebel<br>- Choose this option instead.<br>- YOLO! |
| `song` | If you could listen to only one of these songs for the rest of your life, which would it be?<br>Options:<br>- "Hotline Bling" by Drake<br>- "I Want It That Way" by Backstreet Boys<br>- "Shake It Off" by Taylor Swift<br>- "Baby" by Justin Bieber<br>- "Sandstorm" by Darude<br>- "Hello" by Adele<br>- "Thinking Out Loud" by Ed Sheeran |
| `date` | Pick a day of the year! |
| `pet` | If you could have any animal in the world as a pet, what would it be? |
| `gerald` | Choose your favorite photo of Gerald Friedland! (Options shown under Question 1) |
| `smallest` | Try to guess the smallest unique positive **integer** that anyone will put! |

#### 2. `checkboxes`

Numbers randomly selected from 1–10, 2015, 9000, and 9001, matched to each student. Students may be matched to multiple numbers.

Each row has a `time` column (again serving as a unique identifier) and contains the value `'True'` if the student is matched with that number or `'False'` otherwise.

The column names in this table are the following strings, referring to each possible number:

`'0'`, `'1'`, `'2'`, `'4'`, `'5'`, `'6'`, `'7'`, `'8'`, `'9'`, `'10'`, `'2015'`, `'9000'`, `'9001'`

A time in `students` matches up with a time in `checkboxes`. For example, the row with time `"11/11/2015 9:54:03"` in `students` matches the row with the same time in `checkboxes`.

We use time to uniquely identify each student, rather than using names or email addresses.

> **Note:** If you are looking for your personal response within the data, you may notice that some answers differ slightly from what you originally entered. To make SQLite accept the data and to optimize joins, we applied the following cleaning steps:
>
> - `number` and `smallest`: If no number was entered, we used `-1` as a placeholder.
> - `color` and `pet`: All strings were converted to lowercase.

<hr style="border: 1px solid #fdb515;" />

## What Would SQL print?

Before we start, inspect the schema of the tables that we've created for you. A schema tells you the name of each of our tables and their attributes. In general, you can think of a schema as a map that describes the logical entities and relationships of a database. Just as the outline of a book tells a reader the order and category in which content is organized, a schema details the organizational hierarchy of information within a database.


In [ ]:
# Get the schema of all tables
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()
print("Tables in database:")
for table in tables:
    print(f"\n{table[0]}:")
    cursor.execute(f"SELECT sql FROM sqlite_master WHERE type='table' AND name='{table[0]}'")
    schema = cursor.fetchone()
    if schema and schema[0]:
        print(schema[0])

Let's also take a look at some of the entries in our table. There are a lot of entries though, so let's just output the first 20:


In [ ]:
run_query("SELECT * FROM students LIMIT 20")

If you're curious about some of the answers students put into the Google form, open up `data.sql` in your favorite text editor and take a look!

For each of the SQL queries below, decide to yourself what the query is looking for, then try running the query yourself and see!

> **Note:** You don't need to submit anything for this question, just think/talk about what each query is doing, and then run the query in the code cells below.


**Example 1:** Select all records from students. `*` is shorthand for all columns!

In [ ]:
run_query("SELECT * FROM students")  # This is a comment. * is shorthand for all columns!

**Example 2:** Select the color from students who said their favorite number was 16.


In [ ]:
run_query("SELECT color FROM students WHERE number = 16")

**Example 3:** Select the song and pet from students who said their favorite color was blue and picked December 25th.


In [ ]:
run_query('SELECT song, pet FROM students WHERE color = "blue" AND date = "12/25"')

<hr style="border: 1px solid #fdb515;" />

## Obedience

To warm-up, let's ask a simple question related to our data: Is there a
correlation between whether students do as they're told and their
favorite image of [Gerald Friedland](https://www2.eecs.berkeley.edu/Faculty/Homepages/friedland.html)? (past Data C88C instructor @UC Berkeley)

![gerald](gerald.jpg)

Write a SQL query to create a table that contains the columns `seven` (this
column representing "obedience") and `gerald` (the image students selected)
from the `students` table.

You should get the following output:

```sql
SELECT * FROM obedience LIMIT 10;
7|Option 2
Choose this option instead.|Option 3
YOLO!|Option 3
7|Option 4
7|Option 5
YOLO!|Option 3
Choose this option instead.|Option 3
Choose this option instead.|Option 3
7|Option 1
7|Option 3
```

In [ ]:

run_query("""CREATE TABLE obedience AS
  SELECT * FROM students WHERE False;  -- Replace with your query
""")


run_query("""SELECT * FROM obedience LIMIT 10""")

In [ ]:
grader.check("obedience")

<hr style="border: 1px solid #fdb515;" />

## Go Bears! (And Dogs?)

Now that we have learned how to select columns from a SQL table, let's filter the
results to see some more interesting results!

It turns out that our students have a lot of school spirit: the most popular favorite
color was `'blue'`. You would think that this school spirit would carry over to the
pet answer, and everyone would want a pet bear! Unfortunately, this was not the case,
and the majority of students opted to have a pet `'dog'` instead. That is the more
sensible choice, I suppose...

Write a SQL query to create a table that contains both the column `color` and the
column `pet`, using the keyword `WHERE` to restrict the answers to the most popular
results of color being `'blue'` and pet being `'dog'`.

You should get the following output:

```sql
SELECT * FROM blue_dog;
blue|dog
blue|dog
blue|dog
```

In [ ]:
run_query("""CREATE TABLE blue_dog AS
  SELECT * FROM students WHERE False;  -- Replace with your query
""")



In [ ]:
grader.check("bluedog")

<hr style="border: 1px solid #fdb515;" />

## The Smallest Unique Integer

Who successfully managed to guess the smallest unique integer value? Let's find
out!

To keep it simple, we limited the allowed inputs to be a positive number (greater than zero).

Unfortunately we have not learned how to do aggregations, which can help us count
the number of times a specific value was selected, in SQL just yet. As such, we
can only hand inspect our data to determine it.

Write a SQL query with the columns `time` and `smallest` to try to determine what
the smallest integer value is. In order to make it easier for us to inspect these
values, use `ORDER BY` to sort the numerical values, and `LIMIT` your result to
the first 20 values that are greater than the number 3.

The first 5 lines of your output should look like this:

```sql
SELECT * FROM smallest_int LIMIT 5;
4/17/2019 10:19:17|4
4/19/2019 17:46:44|4
4/20/2019 20:29:22|5
4/16/2019 18:44:34|6
4/17/2019 9:44:12|6
```

In [ ]:
run_query("""CREATE TABLE smallest_int AS
  SELECT * FROM students WHERE False;  -- Replace with your query
""")


run_query("""SELECT * FROM smallest_int LIMIT 5""")

In [ ]:
grader.check("smallest-int")

<hr style="border: 1px solid #fdb515;" />

## Sevens

Let's take a look at data from both of our tables, `students` and `checkboxes`.
Write a SQL query to create a table with just the column `seven` from `students`, filtering
for students who said their favorite number (column `number`) was 7 in the `students`
table and who checked the box for seven (have the value `'True'` in column `'7'`) in the `checkboxes` table.

**Hint:** In order to examine rows from both the `students` and the `checkboxes` table, we will
need to perform a join. How would you specify the `WHERE` clause to make the `SELECT` statement only consider
rows in the joined table whose values all correspond to the same student? If
you find that your output is massive and overwhelming, then you are probably missing
the necessary condition in your `WHERE` clause to ensure this.

> *Note:* The columns in the `checkboxes` table are strings with the associated number,
> so you must put quotes around the column name to refer to it. For example if you alias
> the table as `a`, to get the column to see if a student checked 9001, you must write
> `a.'9001'`.

You should get the following output:

```sql
SELECT * FROM sevens;
7
```

In [ ]:
run_query("""CREATE TABLE sevens AS
  SELECT * FROM students WHERE False;  -- Replace with your query
""")


run_query("""SELECT * FROM sevens""")

In [ ]:
grader.check("sevens")

<hr style="border: 1px solid #fdb515;" />

## Matchmaker, Matchmaker

Did you take C88C with the hope of finding your soul mate? Well you're in luck!
With all this data in hand, it's easy for us to find your perfect match. If two
students want the same pet and have the same taste in music, they are clearly meant
to be together! In order to provide some more information for the potential lovebirds
to converse about, let's include the favorite colors of the two individuals as well!

In order to match up students, you will have to do a join on the `students` table
with itself. When you do a join, SQLite will match every single row with every single
other row, so make sure you do not match anyone with themselves, or match any given pair
twice!

*Hint:* You may want to enforce a sort of "ordering" on the column `time` (which
is a unique identifier) from your joined tables, in order to do the above correctly.

Write a SQL query to create a table that has 4 columns:
- The shared preferred pet of the couple
- The shared favorite song of the couple
- The favorite color of the first person
- The favorite color of the second person

You should get the following output:

```sql
SELECT * FROM matchmaker LIMIT 20;
dog|"I Want it That Way"" by Backstreet Boys|none|blue
dog|"I Want it That Way"" by Backstreet Boys|none|blue
dog|"Hello" by Adele|green|pink
dog|"Hello" by Adele|green|purple
dog|"I Want it That Way"" by Backstreet Boys|blue|blue
dog|"Thinking Out Loud" by Ed Sheeran|don't have one|black
dog|"Thinking Out Loud" by Ed Sheeran|don't have one|black
owl|"Hello" by Adele|blue|blue
dog|"Hello" by Adele|pink|purple
dolphin|"I Want it That Way"" by Backstreet Boys|green|blue
cat|"Hello" by Adele|pink|blue
dog|"Hotline Bling" by Drake|white|blue
dog|"Hotline Bling" by Drake|white|black
dog|"Thinking Out Loud" by Ed Sheeran|black|black
dog|"Hotline Bling" by Drake|blue|black
```

In [ ]:

run_query("""CREATE TABLE matchmaker AS
  SELECT * FROM students WHERE False;  -- Replace with your query
""")


run_query("""SELECT * FROM matchmaker LIMIT 20""")

In [ ]:
grader.check("matchmaker")

<hr style="border: 1px solid #fdb515;" />

## The COUNT aggregate

Recall how finding the smallest integer anyone chose was rather painful, because we could
not simply count up how many times each integer was chosen by anyone.

Bring in SQL aggregation, which is commonly used to aggregate values in order to answer these
types of questions!

In order to perform SQL aggregation, we need to group rows in our table by one or more attributes. Once
we have groups, we can aggregate over the groups in our table and find things like the maximum value (`MAX`),
the minimum value (`MIN`), the number of rows in the group (`COUNT`), the average over all of the values
(`AVG`), and more! `SELECT` statements that use aggregation are marked by two things: an aggregate function
(`MAX`, `MIN`, `COUNT`, `AVG`, etc.) and a `GROUP BY` clause. For example:

```sql
SELECT song, MAX(number) FROM students GROUP BY song;
|69
Baby|76
Hello|100
Hotline Bling|100
Sandstorm|100
Shake It Off|98
That Way|100
Thinking Out Loud|99
```

This `SELECT` statement groups all of the rows in our table `students` by `song`. Then, within each
group, we perform aggregation by MAXing over the attribute `number`. By selecting `song` and `MAX(number)`,
we then can see the highest `number` any student chose for any given `song`.

### The Smallest Unique Integer (Part 2)

Now, let's revisit the previous problem of finding the smallest integer that anyone chose, and take
a closer look at the `COUNT` aggregate.

Just like `MAX` above, we can select a `COUNT` of some attribute in our query after grouping the rows
in our table by some attribute.

Write a SQL query where the first column is the attribute `smallest` and the second column is the number
of times that number was chosen by a student (hint: use the `COUNT` aggregate). In order to cut out the
people who chose not to respond, and the sneaky cheaters that tried to put small non-integer values,
filter your results to only include rows where `smallest` is greater than or equal to 1!

*Hint:* You may find that there isn't a particular attribute you should have to perform the `COUNT`
aggregation over. If you are only interested in counting the number of rows in a group, you can just
say `COUNT(*)`.

*Hint:* Think about what attribute you need to `GROUP BY`.

After you've defined your table, you should get something like:

```
SELECT * FROM smallest_int_count LIMIT 10;
1|34
2|4
3|7
4|2
5|1
6|2
7|5
8|1
10|1
11|1
```

It looks like the number `5` is the smallest unique integer!

In [ ]:

run_query("""CREATE TABLE smallest_int_count AS
  SELECT * FROM students WHERE False;  -- Replace with your query
""")


run_query("""SELECT * FROM smallest_int_count LIMIT 10""")

In [ ]:
grader.check("smallest-int-count")